# Домашнее задание № 1 — Анализ  сообщений на спам

**ФИО:** Яцкий Максим Евгеньевич
**Группа:** РИ-431003

В этой работе я посмотрю, чем спам отличается от обычных сообщений. Для этого использовал Python, pandas, matplotlib и seaborn

## 1. Постановка задачи

**Бизнес-задача.**  Необходимо сделать программу, которая сможет отличать спам от обычных сообщений. Это может помочь автоматически отправлять рекламу и подозрительные сообщения в отдельную папку.

**ML-задача.** Это **бинарная классификация**: у сообщения может быть один из двух классов — **«Спам» (1)** или **«Обычное» (0)**. Позже модель будет получать текст и определять его класс.

**Данные.** Для работы использую открытый набор русскоязычных сообщений [benzlokzik/russian-spam-fork](https://huggingface.co/datasets/benzlokzik/russian-spam-fork). В нём есть текст (`text`) и метка класса (`label`). Датасет большой, поэтому для первого анализа возьму случайную выборку из **15 000 сообщений**. Это сообщения разных типов, не обязательно только SMS

## 2. Загрузка данных

Код ниже скачивает данные автоматически в Google Colab. Для этого нужен интернет. Мы используем открытую копию набора, поэтому логин Hugging Face не требуется.


In [ ]:
!pip -q install datasets pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from itertools import islice
from datasets import load_dataset
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_colwidth", 110)

sample_size = 15000

# Загружаем датасет частями, не скачивая большое кол-во сообщений сразу
dataset_stream = load_dataset("benzlokzik/russian-spam-fork", split="train", streaming=True)
dataset_stream = dataset_stream.shuffle(seed=42, buffer_size=10000)
rows = list(islice(dataset_stream, sample_size))
df = pd.DataFrame(rows)

# Проверяем и приводим названия к одному виду
assert {"text", "label"}.issubset(df.columns), f"Неожиданные столбцы: {df.columns.tolist()}"
df = df[["text", "label"]].copy()
df['label'] = df['label'].astype(int)
assert df['label'].isin([0, 1]).all(), "Обнаружены неизвестные классы"
df['category'] = df['label'].map({0: "Обычное", 1: "Спам"})
print("Загружено сообщений:", len(df))
display(df[["text", "category"]].head(10).rename(columns={"text": "Текст сообщения", "category": "Категория"}))

**Что здесь происходит:**  загрузили случайную часть набора данных и показал первые 10 сообщений. В таблице есть текст и категория. Числа 0 и 1 заменил понятными названиями.

## 3. Размер таблицы, пропуски и повторы

Сначала посмотрю, сколько у меня строк, нет ли пустых сообщений и одинаковых текстов.

In [ ]:
summary_df = pd.DataFrame({
    "Показатель": ["Всего сообщений", "Столбцов", "Пустых текстов", "Повторяющихся текстов"],
    "Количество": [len(df), len(df.columns),
                   df['text'].isna().sum(),
                   df['text'].duplicated().sum()]
})
display(summary_df)
print("Типы столбцов:")
display(pd.DataFrame({"Столбец": df.dtypes.index, "Тип данных": df.dtypes.astype(str).values}))

In [ ]:
# Убираем пустые и повторяющиеся сообщения, чтобы они не мешали анализ
rows_before = len(df)
df = df.dropna(subset=['text']).copy()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ""].copy()

# Если один и тот же текст имеет разные метки, это стоит отдельно проверить
conflicting_labels = df.groupby('text')['label'].nunique()
print("Текстов с противоречивыми метками:", int((conflicting_labels > 1).sum()))
df = df.drop_duplicates(subset=['text'], keep="first").copy()
print(f"Удалено пустых и повторяющихся строк: {rows_before - len(df)}")
print(f"Осталось сообщений: {len(df)}")

повторы лучше убрать, иначе одни и те же тексты могут слишком сильно влиять на результат. Если у одинаковых текстов разные метки, это отдельная проблема качества данных — при будущем обучении её нужно изучить.

## 4. Сколько спама и обычных сообщений

In [ ]:
class_counts = df['category'].value_counts().reindex(["Обычное", "Спам"], fill_value=0)
class_summary = pd.DataFrame({'category': class_counts.index,
                                 "Количество": class_counts.values,
                                 "Доля, %": (100 * class_counts.values / len(df)).round(1)})
display(class_summary.rename(columns={"category": "Категория"}))
ax = sns.barplot(data=class_summary, x='category', y="Количество", color="#4682B4")
ax.set(title="Сколько обычных сообщений и спама", xlabel="Категория сообщения", ylabel="Количество сообщений")
for i, v in enumerate(class_summary["Количество"]): ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.show()
print("Вывод:", "Спама меньше, чем обычных сообщений." if class_counts["Спам"] < class_counts["Обычное"] else "Спама не меньше, чем обычных сообщений.")
print("Разница между классами важна: при обучении нельзя судить о модели только по доле правильных ответов.")

## 5. Длина сообщений

проверим, отличаются ли классы по числу символов и слов.

In [ ]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
length_stats = df.groupby('category')[['char_count', 'word_count']].agg(["mean", "median", "min", "max"]).round(1)
length_stats.columns = [f"{x} — {dict(mean='среднее', median='медиана', min='минимум', max='максимум')[y]}" for x,y in length_stats.columns]
display(length_stats.rename(columns=lambda col: col.replace("char_count", "Длина в символах").replace("word_count", "Количество слов")))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x='char_count', hue='category', bins=45, stat="density", common_norm=False, ax=axes[0])
axes[0].set(xlabel="Длина сообщения (символы)", ylabel="Плотность", title="Длина сообщений")
axes[0].set_xlim(0, df['char_count'].quantile(.98))
sns.boxplot(data=df, x='category', y='word_count', ax=axes[1], color="#8bbad9", showfliers=False)
axes[1].set(xlabel="Категория", ylabel="Количество слов", title="Сколько слов в сообщениях")
plt.tight_layout(); plt.show()
median_lengths = df.groupby('category')['char_count'].median()
print("Медианная длина обычных сообщений:", round(median_lengths.get("Обычное", float('nan')),1))
print("Медианная длина спама:", round(median_lengths.get("Спам", float('nan')),1))
print("Вывод: длина может пригодиться как дополнительный признак, но по ней одной нельзя определить спам.")

## 6. Ссылки, цифры и восклицательные знаки

В рекламных сообщениях часто бывают ссылки, номера телефонов или призывы что-то сделать. Проверю, насколько часто эти признаки встречаются в каждой категории.

In [ ]:
text_lower = df['text'].str.lower()
df['has_link'] = text_lower.str.contains(r"https?://|www\.|(?:[a-z0-9-]+\.)+(?:ru|com|рф|net|org)(?:/|\b)", regex=True)
df['has_digits'] = text_lower.str.contains(r"\d", regex=True)
df['has_exclamation'] = text_lower.str.contains("!", regex=False)
feature_cols = ['has_link', 'has_digits', 'has_exclamation']
feature_rates = (df.groupby('category')[feature_cols].mean() * 100).round(1)
feature_rates.index.name = 'category'
display(feature_rates.rename_axis("Категория").rename(columns={"has_link": "Со ссылкой, %", "has_digits": "С цифрами, %", "has_exclamation": "С восклицанием, %"}))
feature_plot_df = feature_rates.reset_index().melt(id_vars='category', var_name="Признак", value_name="Доля сообщений, %")
ax = sns.barplot(data=feature_plot_df, x="Признак", y="Доля сообщений, %", hue='category')
ax.set(title="Как часто встречаются разные признаки", xlabel="Признак", ylabel="Доля сообщений, %")
ax.set_xticks(range(3), ["Есть ссылка", "Есть цифры", "Есть !"])
plt.tight_layout(); plt.show()
print("Вывод: признаки могут быть полезны для будущей модели, но ни один не доказывает, что сообщение — спам.")

## 7. Частые слова

Посчитаем самые частые слова в каждой категории. Уберем короткие слова и часть служебных слов, например и,на, это и так далее

In [ ]:
stop_words = set("и в во не что он она оно на я с со как а то все так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про просто очень тебя ваши наш ваша вас это вам".split())
def get_top_words(text_series):
    words=[]
    for s in text_series.dropna().str.lower():
        words.extend(w for w in re.findall(r"[а-яё]{3,}", s) if w not in stop_words)
    return Counter(words).most_common(12)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, category in zip(axes, ["Обычное", "Спам"]):
    top_words = get_top_words(df.loc[df['category'] == category, 'text'])
    if top_words:
        words, frequencies = zip(*top_words)
        sns.barplot(x=list(frequencies), y=list(words), ax=ax, color="#4682B4")
    ax.set(title=f"Частые слова: {category.lower()}", xlabel="Сколько раз встретилось слово", ylabel="Слово")
plt.tight_layout(); plt.show()
print("Вывод: наборы популярных слов можно сравнивать, но частое слово не означает спам.")

## 8. Связь числовых признаков

Посмотрим, как связаны длина, число слов, ссылки, цифры и класс сообщения.

In [ ]:
numeric_features = df[['label', 'char_count', 'word_count', 'has_link', 'has_digits', 'has_exclamation']].copy()
numeric_features = numeric_features.rename(columns={'label': "Спам (1 — да)", 'has_link': "Наличие ссылки", 'has_digits': "Наличие цифр", 'has_exclamation': "Наличие !", "char_count": "Длина в символах", "word_count": "Количество слов"})
corr_matrix = numeric_features.astype(float).corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Связь признаков между собой")
plt.tight_layout(); plt.show()
print("Вывод: значения ближе к 1 или -1 показывают более сильную связь. Это не значит, что один признак вызывает другой.")

## 9. Итоговые выводы

После анализа я сделал следующие выводы:

1. В наборе есть тексты сообщений и отметка, является ли сообщение спамом. Поэтому его можно использовать для задачи классификации.
2. Я посмотрел, сколько сообщений каждого класса, есть ли пустые и повторяющиеся строки.
3. Длина сообщения, число слов, ссылки и цифры могут помочь при определении спама. Но по одному признаку делать вывод тяжело.
4. Полезно учитывать слова, которые чаще встречаются в разных классах.



**Источники:** [датасет](https://huggingface.co/datasets/benzlokzik/russian-spam-fork)